<a href="https://colab.research.google.com/github/Maddox159-crypto/2026_ESAA_assignment/blob/main/RNN_%EC%8B%A4%EC%8A%B5_%EC%BD%94%EB%93%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[개념]

## 문자 단위 RNN이란?

- RNN의 입력과 출력 단위를 단어가 아니라 문자로 바꾼 것
- RNN 구조 자체는 그대로이고, 입/출력 단위만 문자로 바뀜
- 이번 실습은 다대다 구조로 구현: 매 시점마다 문자 하나를 입력받아 다음 문자를 예측함

**공통 처리 흐름**
1. **문자 집합 생성**: 중복을 제거한 문자들의 집합을 만듦
2. **정수 인코딩**: 각 문자에 고유한 정수를 부여함 (예측 결과를 문자로 되돌리려고 정수 → 문자 사전도 만듦)
3. **원-핫 인코딩**: 각 문자를 원-핫 벡터로 바꿔 입력으로 사용함. 그래서 **입력의 크기 = 문자 집합의 크기**
4. **텐서 변환**: 파이토치 RNN은 3차원 텐서 (배치 크기, 시점 수, 입력 크기)를 입력으로 받음
5. **모델 구성**: RNN 층과 출력층(완전연결층)을 연결
6. **손실 계산**: 출력 (배치, 시점, 출력 크기)와 레이블 (배치, 시점)을 펼쳐서 배치 차원과 시점 차원을 하나로 합친 뒤 크로스 엔트로피 손실을 계산함

---

##  실습 1: apple → pple!

- **목표**: 문자 시퀀스 apple을 입력하면 pple!를 출력하는 RNN. 실용적인 의미보다는 RNN의 동작을 이해하기 위한 예제
- **문자 집합**: !, a, e, l, p → 총 5개
- **데이터**
  - 입력: a, p, p, l, e → 정수로 [1, 4, 4, 3, 2]
  - 레이블: p, p, l, e, ! → 정수로 [4, 4, 3, 2, 0]
  - 입력과 레이블은 한 칸씩 밀린(shift) 관계 → 각 시점에서 **다음 문자를 예측**함
- **텐서 크기**: 입력 (1, 5, 5) = 배치 1개, 시점 5개, 원-핫 크기 5 / 레이블 (1, 5)
- **하이퍼파라미터**: 은닉 상태 크기 5, 출력 크기 5, 학습률 0.1
- **모델**: RNN 층 1개 + 출력층
- **학습**: 크로스 엔트로피 손실, Adam 옵티마이저, 100 에포크
- **결과**: 처음에는 엉뚱한 문자를 예측하다가(예: pp!p!) 학습이 끝나면 **pple!**를 정확히 예측함

---

## 실습 2: 더 많은 데이터로 학습한 Char RNN

- **데이터**: 영어 문장 하나 ("if you want to build a ship, ... of the sea.")
- **문자 집합**: 공백과 문장 부호를 포함해 총 25개
- **샘플 만들기 (슬라이딩 윈도우)**
  - 문장을 **길이 10(sequence length)** 단위로 한 칸씩 이동하며 자름
  - 입력: i번째부터 10글자 / 레이블: i+1번째부터 10글자 (한 칸씩 밀린 시퀀스)
  - 예) "if you wan" → "f you want"
  - 이렇게 총 **170개 샘플** 생성
- **텐서 크기**: 입력 (170, 10, 25) / 레이블 (170, 10)
- **하이퍼파라미터**: 은닉 상태 크기 = 문자 집합 크기 25 (다른 값도 가능), 학습률 0.1
- **모델**: 실습 1과 거의 같지만 **은닉층을 2개 쌓음** (RNN의 층 수 인자를 2로 설정 → 적층 RNN)
- **손실 계산**: 출력을 (1700, 25), 레이블을 (1700)으로 펼쳐서 계산
- **예측 문장 복원**: 첫 샘플은 예측 결과를 전부 쓰고, 이후 샘플은 마지막 글자만 하나씩 이어 붙여 전체 문장을 만듦
- **결과**: 처음에는 의미 없는 문자열을 내다가, 마지막 에포크에서는 원래 문장과 거의 같은 문장을 생성함

[필사]

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [2]:
input_str = 'apple'
label_str = 'pple!'
char_vocab = sorted(list(set(input_str+label_str)))
vocab_size = len(char_vocab)
print('문자 집합의 크기 : {}'.format(vocab_size))

문자 집합의 크기 : 5


In [3]:
input_size = vocab_size # 입력의 크기는 문자 집합의 크기
hidden_size = 5
output_size = 5
learning_rate = 0.1

In [4]:
char_to_index = dict((c, i) for i, c in enumerate(char_vocab)) # 문자에 고유한 정수 인덱스 부여
print(char_to_index)

{'!': 0, 'a': 1, 'e': 2, 'l': 3, 'p': 4}


In [5]:
index_to_char = {}
for key, value in char_to_index.items():
    index_to_char[value] = key
print(index_to_char)

{0: '!', 1: 'a', 2: 'e', 3: 'l', 4: 'p'}


In [6]:
x_data = [char_to_index[c] for c in input_str]
y_data = [char_to_index[c] for c in label_str]
print(x_data)
print(y_data)

[1, 4, 4, 3, 2]
[4, 4, 3, 2, 0]


In [7]:
# 배치 차원 추가
# 텐서 연산인 unsqueeze(0)를 통해 해결할 수도 있었음.
x_data = [x_data]
y_data = [y_data]
print(x_data)
print(y_data)

[[1, 4, 4, 3, 2]]
[[4, 4, 3, 2, 0]]


In [8]:
x_one_hot = [np.eye(vocab_size)[x] for x in x_data]
print(x_one_hot)

[array([[0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 1.],
       [0., 0., 0., 1., 0.],
       [0., 0., 1., 0., 0.]])]


In [9]:
X = torch.FloatTensor(x_one_hot)
Y = torch.LongTensor(y_data)

/tmp/ipykernel_747/2348034151.py:1: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:252.)
  X = torch.FloatTensor(x_one_hot)


In [10]:
X = torch.FloatTensor(x_one_hot)
Y = torch.LongTensor(y_data)

In [11]:
print('훈련 데이터의 크기 : {}'.format(X.shape))
print('레이블의 크기 : {}'.format(Y.shape))

훈련 데이터의 크기 : torch.Size([1, 5, 5])
레이블의 크기 : torch.Size([1, 5])


In [12]:
class Net(torch.nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(Net, self).__init__()
        self.rnn = torch.nn.RNN(input_size, hidden_size, batch_first=True) # RNN 셀 구현
        self.fc = torch.nn.Linear(hidden_size, output_size, bias=True) # 출력층 구현

    def forward(self, x): # 구현한 RNN 셀과 출력층을 연결
        x, _status = self.rnn(x)
        x = self.fc(x)
        return x

In [13]:
net = Net(input_size, hidden_size, output_size)

In [14]:
outputs = net(X)
print(outputs.shape) # 3차원 텐서

torch.Size([1, 5, 5])


In [15]:
print(outputs.view(-1, input_size).shape) # 2차원 텐서로 변환

torch.Size([5, 5])


In [16]:
print(Y.shape)
print(Y.view(-1).shape)

torch.Size([1, 5])
torch.Size([5])


In [17]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), learning_rate)

In [18]:
for i in range(100):
    optimizer.zero_grad()
    outputs = net(X)
    loss = criterion(outputs.view(-1, input_size), Y.view(-1)) # view를 하는 이유는 Batch 차원 제거를 위해
    loss.backward() # 기울기 계산
    optimizer.step() # 아까 optimizer 선언 시 넣어둔 파라미터 업데이트

    # 아래 세 줄은 모델이 실제 어떻게 예측했는지를 확인하기 위한 코드.
    result = outputs.data.numpy().argmax(axis=2) # 최종 예측값인 각 time-step 별 5차원 벡터에 대해서 가장 높은 값의 인덱스를 선택
    result_str = ''.join([index_to_char[c] for c in np.squeeze(result)])
    print(i, "loss: ", loss.item(), "prediction: ", result, "true Y: ", y_data, "prediction str: ", result_str)

0 loss:  1.638728141784668 prediction:  [[3 3 3 3 3]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  lllll
1 loss:  1.467581033706665 prediction:  [[3 3 3 0 0]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  lll!!
2 loss:  1.3307170867919922 prediction:  [[3 2 2 2 0]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  leee!
3 loss:  1.1813883781433105 prediction:  [[4 4 4 2 0]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  pppe!
4 loss:  1.0075994729995728 prediction:  [[4 4 4 2 0]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  pppe!
5 loss:  0.8543550372123718 prediction:  [[4 4 4 2 0]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  pppe!
6 loss:  0.6960480809211731 prediction:  [[4 4 4 2 0]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  pppe!
7 loss:  0.5501729249954224 prediction:  [[4 4 3 2 0]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  pple!
8 loss:  0.4146324098110199 prediction:  [[4 4 3 2 0]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  pple!
9 loss:  0.2949022054672241 prediction:  [[4 4 3 2 0]] tr

In [19]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [20]:
sentence = ("if you want to build a ship, don't drum up people together to "
            "collect wood and don't assign them tasks and work, but rather "
            "teach them to long for the endless immensity of the sea.")

In [21]:
char_set = list(set(sentence)) # 중복을 제거한 문자 집합 생성
char_dic = {c: i for i, c in enumerate(char_set)} # 각 문자에 정수 인코딩
print(char_dic) # 공백도 여기서는 하나의 원소

{'d': 0, 'h': 1, 'c': 2, "'": 3, 's': 4, 'o': 5, 'k': 6, 'l': 7, 'u': 8, 'a': 9, ' ': 10, 'p': 11, 'f': 12, 'n': 13, '.': 14, 'm': 15, 'w': 16, 'r': 17, 'y': 18, 't': 19, 'g': 20, 'e': 21, 'b': 22, 'i': 23, ',': 24}


In [22]:
dic_size = len(char_dic)
print('문자 집합의 크기 : {}'.format(dic_size))

문자 집합의 크기 : 25


In [23]:
# 하이퍼파라미터 설정
hidden_size = dic_size
sequence_length = 10 # 임의 숫자 지정
learning_rate = 0.1

In [24]:
# 데이터 구성
x_data = []
y_data = []

for i in range(0, len(sentence) - sequence_length):
    x_str = sentence[i:i + sequence_length]
    y_str = sentence[i + 1: i + sequence_length + 1]
    print(i, x_str, '->', y_str)

    x_data.append([char_dic[c] for c in x_str])  # x str to index
    y_data.append([char_dic[c] for c in y_str])  # y str to index

0 if you wan -> f you want
1 f you want ->  you want 
2  you want  -> you want t
3 you want t -> ou want to
4 ou want to -> u want to 
5 u want to  ->  want to b
6  want to b -> want to bu
7 want to bu -> ant to bui
8 ant to bui -> nt to buil
9 nt to buil -> t to build
10 t to build ->  to build 
11  to build  -> to build a
12 to build a -> o build a 
13 o build a  ->  build a s
14  build a s -> build a sh
15 build a sh -> uild a shi
16 uild a shi -> ild a ship
17 ild a ship -> ld a ship,
18 ld a ship, -> d a ship, 
19 d a ship,  ->  a ship, d
20  a ship, d -> a ship, do
21 a ship, do ->  ship, don
22  ship, don -> ship, don'
23 ship, don' -> hip, don't
24 hip, don't -> ip, don't 
25 ip, don't  -> p, don't d
26 p, don't d -> , don't dr
27 , don't dr ->  don't dru
28  don't dru -> don't drum
29 don't drum -> on't drum 
30 on't drum  -> n't drum u
31 n't drum u -> 't drum up
32 't drum up -> t drum up 
33 t drum up  ->  drum up p
34  drum up p -> drum up pe
35 drum up pe -> rum up peo
36

In [25]:
print(x_data[0])
print(y_data[0])

[23, 12, 10, 18, 5, 8, 10, 16, 9, 13]
[12, 10, 18, 5, 8, 10, 16, 9, 13, 19]


In [26]:
x_one_hot = [np.eye(dic_size)[x] for x in x_data] # x 데이터는 원-핫 인코딩
X = torch.FloatTensor(x_one_hot)
Y = torch.LongTensor(y_data)

In [27]:
print('훈련 데이터의 크기 : {}'.format(X.shape))
print('레이블의 크기 : {}'.format(Y.shape))

훈련 데이터의 크기 : torch.Size([170, 10, 25])
레이블의 크기 : torch.Size([170, 10])


In [28]:
print(X[0])

tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         1., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 1.,

In [29]:
print(Y[0])

tensor([12, 10, 18,  5,  8, 10, 16,  9, 13, 19])


In [30]:
class Net(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, layers): # 현재 hidden_size는 dic_size와 같음.
        super(Net, self).__init__()
        self.rnn = torch.nn.RNN(input_dim, hidden_dim, num_layers=layers, batch_first=True)
        self.fc = torch.nn.Linear(hidden_dim, hidden_dim, bias=True)

    def forward(self, x):
        x, _status = self.rnn(x)
        x = self.fc(x)
        return x

In [31]:
net = Net(dic_size, hidden_size, 2) # 이번에는 층을 두 개 쌓습니다.

In [32]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), learning_rate)

In [33]:
outputs = net(X)
print(outputs.shape) # 3차원 텐서

torch.Size([170, 10, 25])


In [34]:
print(outputs.view(-1, dic_size).shape) # 2차원 텐서로 변환

torch.Size([1700, 25])


In [35]:
print(Y.shape)
print(Y.view(-1).shape)

torch.Size([170, 10])
torch.Size([1700])


In [36]:
for i in range(100):
    optimizer.zero_grad()
    outputs = net(X) # (170, 10, 25) 크기를 가진 텐서를 매 에포크마다 모델의 입력으로 사용
    loss = criterion(outputs.view(-1, dic_size), Y.view(-1))
    loss.backward()
    optimizer.step()

    # results의 텐서 크기는 (170, 10)
    results = outputs.argmax(dim=2)
    predict_str = ""
    for j, result in enumerate(results):
        if j == 0: # 처음에는 예측 결과를 전부 가져오지만
            predict_str += ''.join([char_set[t] for t in result])
        else: # 그 다음에는 마지막 글자만 반복 추가
            predict_str += char_set[result[-1]]

    print(predict_str)

uu'sus'u'gu'uu' su' ''sug''''u'''u' usu'gu'uu'''u' uusuuu'suu''u''u'g'us'u''gu' u''u''gu''ssuuug'u' u s'gu'us'u'' su'uuuuu'suu''u'uuu''uu'''u'suuusuuu'uu''uu 'uu'u'guu 'uu'uuu'gu'
                                                                                                                                                                                   
 ,,t s,,t tt tt  t  ttt   t   ttt shtt  stt  tt  t tt  ttt  tt    tttt tttttt   ttt ttt   tt ttt  tt  ttttt   ttt t t ttt tt  tt ttt   ttt  ttt ttt   t ttt  tt t  ttt tt  t t t tt
   e  e  ee ee eoeo e ee o eoeeoneo e eo et o eo o e ee o eee eeeo eoeeeoeo e ee  eeoeeeee eee eo e e eeeenee oee ee  e ee eee eeee ene e eeeoeeoee eeee eo eo eeeoeeeeeeeeo e ee e
 eotmotmtmmm mmemmemtmo o memee mmemm m mtmoemtem memo memtmemmmeemtmeeoememememmtmmemm m mtmemmememtmemm memmmmemm m mmm m mem meemtmmemmtmeemtmmememm m o m memmmmmememmmmmemm m 
 m t tmtot t tttt t o t tottt t t t t totm t tottoto t t t to t t to t t tt t t t t t t tmto t t t t